# Time Series Forecasting

End-to-end pipeline for time series forecasting using **CatBoost** and **LightGBM** with forward-chaining cross-validation.

## Imports

In [1]:
import os
import gc
import random
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from catboost import CatBoostRegressor

warnings.filterwarnings('ignore')

## Configuration

In [2]:
SEED          = 42
N_SPLITS      = 5
MIN_TRAIN_FRAC = 0.50

TRAIN_PATH = 'train.parquet'
TEST_PATH  = 'test.parquet'

ID_COL     = "id"
TIME_COL   = "ts_index"
WEIGHT_COL = "weight"
TARGET_COL = "y_target"

CAT_COLS = ["code", "sub_code", "sub_category", "horizon"]

In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(SEED)

## Load Data

In [4]:
train = pd.read_parquet(TRAIN_PATH)
test  = pd.read_parquet(TEST_PATH)

print("train:", train.shape)
print("test: ", test.shape)
print("columns:", train.columns.tolist())

train: (5337414, 94)
test:  (1447107, 92)
columns: ['id', 'code', 'sub_code', 'sub_category', 'horizon', 'ts_index', 'feature_a', 'feature_b', 'feature_c', 'feature_d', 'feature_e', 'feature_f', 'feature_g', 'feature_h', 'feature_i', 'feature_j', 'feature_k', 'feature_l', 'feature_m', 'feature_n', 'feature_o', 'feature_p', 'feature_q', 'feature_r', 'feature_s', 'feature_t', 'feature_u', 'feature_v', 'feature_w', 'feature_x', 'feature_y', 'feature_z', 'feature_aa', 'feature_ab', 'feature_ac', 'feature_ad', 'feature_ae', 'feature_af', 'feature_ag', 'feature_ah', 'feature_ai', 'feature_aj', 'feature_ak', 'feature_al', 'feature_am', 'feature_an', 'feature_ao', 'feature_ap', 'feature_aq', 'feature_ar', 'feature_as', 'feature_at', 'feature_au', 'feature_av', 'feature_aw', 'feature_ax', 'feature_ay', 'feature_az', 'feature_ba', 'feature_bb', 'feature_bc', 'feature_bd', 'feature_be', 'feature_bf', 'feature_bg', 'feature_bh', 'feature_bi', 'feature_bj', 'feature_bk', 'feature_bl', 'feature_bm',

In [5]:
train_only = sorted(set(train.columns) - set(test.columns))
test_only  = sorted(set(test.columns)  - set(train.columns))

print("cols only in train:", train_only)
print("cols only in test: ", test_only)

cols only in train: ['weight', 'y_target']
cols only in test:  []


In [6]:
DROP_COLS    = [ID_COL, TARGET_COL, WEIGHT_COL]
FEATURE_COLS = [c for c in train.columns if c not in DROP_COLS]

print(f"{len(FEATURE_COLS)} features")
print(FEATURE_COLS[:20])

91 features
['code', 'sub_code', 'sub_category', 'horizon', 'ts_index', 'feature_a', 'feature_b', 'feature_c', 'feature_d', 'feature_e', 'feature_f', 'feature_g', 'feature_h', 'feature_i', 'feature_j', 'feature_k', 'feature_l', 'feature_m', 'feature_n', 'feature_o']


## Evaluation Metric

Higher score is better (range 0–1). We use a weighted RMSE normalized by the target scale.

## Cross-Validation Splits

Forward-chaining splits ensure the model always trains on past data and validates on future data.

In [7]:
def weighted_rmse_score(y_true, y_pred, weights):
    y_true  = np.asarray(y_true,   dtype=np.float64)
    y_pred  = np.asarray(y_pred,   dtype=np.float64)
    weights = np.asarray(weights,  dtype=np.float64)

    numerator   = np.sum(weights * (y_true - y_pred) ** 2)
    denominator = np.sum(weights * y_true ** 2)

    ratio = numerator / max(denominator, 1e-12)
    ratio = np.clip(ratio, 0.0, 1.0)

    return float(1.0 - np.sqrt(ratio))

In [8]:
def make_forward_splits(ts_index, n_splits=5, min_train_frac=0.5):
    ts = pd.Series(ts_index)
    unique_times = np.array(sorted(ts.unique()))
    n = len(unique_times)

    start = max(int(n * min_train_frac), 1)
    cut_points = np.linspace(start, n, n_splits + 1, dtype=int)

    splits = []
    for i in range(n_splits):
        train_times = unique_times[:cut_points[i]]
        valid_times = unique_times[cut_points[i]:cut_points[i + 1]]

        if len(valid_times) == 0:
            continue

        tr_idx = np.where(ts.isin(train_times))[0]
        va_idx = np.where(ts.isin(valid_times))[0]

        if len(tr_idx) == 0 or len(va_idx) == 0:
            continue

        splits.append((tr_idx, va_idx, train_times, valid_times))

    return splits


splits = make_forward_splits(train[TIME_COL], n_splits=N_SPLITS, min_train_frac=MIN_TRAIN_FRAC)

for i, (tr_idx, va_idx, tr_t, va_t) in enumerate(splits):
    print(f"fold {i}  train={len(tr_idx):,} rows ({tr_t.min()}..{tr_t.max()})  "
          f"valid={len(va_idx):,} rows ({va_t.min()}..{va_t.max()})")

fold 0  train=2,287,193 rows (1..1800)  valid=589,573 rows (1801..2160)
fold 1  train=2,876,766 rows (1..2160)  valid=617,233 rows (2161..2520)
fold 2  train=3,493,999 rows (1..2520)  valid=627,750 rows (2521..2880)
fold 3  train=4,121,749 rows (1..2880)  valid=608,195 rows (2881..3240)
fold 4  train=4,729,944 rows (1..3240)  valid=607,470 rows (3241..3601)


## Model Training Functions

In [9]:
def breakdown_by_group(df, y_true, y_pred, weight_col="weight", group_cols=("horizon", "sub_category")):
    df = df.copy()
    df["_y_true"] = y_true
    df["_y_pred"] = y_pred

    rows = []
    for col in group_cols:
        if col not in df.columns:
            continue
        for key, group in df.groupby(col, observed=True):
            score = weighted_rmse_score(group["_y_true"], group["_y_pred"], group[weight_col])
            rows.append({"group_col": col, "group_value": key, "n_rows": len(group), "score": score})

    return pd.DataFrame(rows).sort_values(["group_col", "score"], ascending=[True, False])

In [10]:
def train_catboost_cv(train_df, test_df, feature_cols, cat_cols, target_col, weight_col, splits):
    oof_pred   = np.zeros(len(train_df))
    test_pred  = np.zeros(len(test_df))
    fold_scores = []
    breakdowns  = []

    valid_cats = [c for c in cat_cols if c in feature_cols]

    for fold, (tr_idx, va_idx, _, _) in enumerate(splits):
        X_tr = train_df.iloc[tr_idx][feature_cols].copy()
        X_va = train_df.iloc[va_idx][feature_cols].copy()
        X_te = test_df[feature_cols].copy()

        for c in valid_cats:
            X_tr[c] = X_tr[c].astype(str)
            X_va[c] = X_va[c].astype(str)
            X_te[c] = X_te[c].astype(str)

        y_tr = train_df.iloc[tr_idx][target_col].values
        w_tr = train_df.iloc[tr_idx][weight_col].values
        y_va = train_df.iloc[va_idx][target_col].values
        w_va = train_df.iloc[va_idx][weight_col].values

        model = CatBoostRegressor(
            loss_function='RMSE',
            iterations=3000,
            learning_rate=0.03,
            depth=8,
            l2_leaf_reg=5.0,
            random_seed=SEED,
            verbose=200
        )

        model.fit(
            X_tr, y_tr,
            cat_features=valid_cats,
            sample_weight=w_tr,
            eval_set=(X_va, y_va),
            use_best_model=True
        )

        va_pred = model.predict(X_va)
        te_pred = model.predict(X_te)

        oof_pred[va_idx]  = va_pred
        test_pred        += te_pred / len(splits)

        score = weighted_rmse_score(y_va, va_pred, w_va)
        fold_scores.append(score)
        print(f"  fold {fold} -> {score:.4f}")

        bd = breakdown_by_group(train_df.iloc[va_idx], y_va, va_pred, weight_col)
        bd["fold"] = fold
        breakdowns.append(bd)

        del X_tr, X_va, X_te, model
        gc.collect()

    oof_score = weighted_rmse_score(train_df[target_col], oof_pred, train_df[weight_col])
    print(f"\nmean={np.mean(fold_scores):.4f}  std={np.std(fold_scores):.4f}  oof={oof_score:.4f}")

    return {
        "oof_pred":      oof_pred,
        "test_pred":     test_pred,
        "fold_scores":   fold_scores,
        "overall_score": oof_score,
        "breakdown":     pd.concat(breakdowns, ignore_index=True) if breakdowns else pd.DataFrame()
    }

In [11]:
def train_lightgbm_cv(train_df, test_df, feature_cols, cat_cols, target_col, weight_col, splits):
    oof_pred   = np.zeros(len(train_df))
    test_pred  = np.zeros(len(test_df))
    fold_scores = []
    breakdowns  = []

    valid_cats = [c for c in cat_cols if c in feature_cols]

    params = {
        "objective":         "regression",
        "metric":            "rmse",
        "boosting_type":     "gbdt",
        "learning_rate":     0.03,
        "num_leaves":        63,
        "min_child_samples": 100,
        "subsample":         0.8,
        "subsample_freq":    1,
        "colsample_bytree":  0.8,
        "reg_alpha":         0.1,
        "reg_lambda":        1.0,
        "seed":              SEED,
        "verbosity":         -1,
        "n_estimators":      5000,
    }

    for fold, (tr_idx, va_idx, _, _) in enumerate(splits):
        X_tr = train_df.iloc[tr_idx][feature_cols].copy()
        X_va = train_df.iloc[va_idx][feature_cols].copy()
        X_te = test_df[feature_cols].copy()

        for c in valid_cats:
            X_tr[c] = X_tr[c].astype('category')
            X_va[c] = X_va[c].astype('category')
            X_te[c] = X_te[c].astype('category')

        y_tr = train_df.iloc[tr_idx][target_col].values
        w_tr = train_df.iloc[tr_idx][weight_col].values
        y_va = train_df.iloc[va_idx][target_col].values
        w_va = train_df.iloc[va_idx][weight_col].values

        model = lgb.LGBMRegressor(**params)
        model.fit(
            X_tr, y_tr,
            sample_weight=w_tr,
            eval_set=[(X_va, y_va)],
            eval_sample_weight=[w_va],
            categorical_feature=valid_cats,
            callbacks=[
                lgb.early_stopping(200, verbose=True),
                lgb.log_evaluation(200)
            ]
        )

        va_pred = model.predict(X_va, num_iteration=model.best_iteration_)
        te_pred = model.predict(X_te, num_iteration=model.best_iteration_)

        oof_pred[va_idx]  = va_pred
        test_pred        += te_pred / len(splits)

        score = weighted_rmse_score(y_va, va_pred, w_va)
        fold_scores.append(score)
        print(f"  fold {fold} -> {score:.4f}")

        bd = breakdown_by_group(train_df.iloc[va_idx], y_va, va_pred, weight_col)
        bd["fold"] = fold
        breakdowns.append(bd)

        del X_tr, X_va, X_te, model
        gc.collect()

    oof_score = weighted_rmse_score(train_df[target_col], oof_pred, train_df[weight_col])
    print(f"\nmean={np.mean(fold_scores):.4f}  std={np.std(fold_scores):.4f}  oof={oof_score:.4f}")

    return {
        "oof_pred":      oof_pred,
        "test_pred":     test_pred,
        "fold_scores":   fold_scores,
        "overall_score": oof_score,
        "breakdown":     pd.concat(breakdowns, ignore_index=True) if breakdowns else pd.DataFrame()
    }

## Train Models

### CatBoost

In [ ]:
cat_result = train_catboost_cv(
    train_df=train,
    test_df=test,
    feature_cols=FEATURE_COLS,
    cat_cols=CAT_COLS,
    target_col=TARGET_COL,
    weight_col=WEIGHT_COL,
    splits=splits
)

0:	learn: 0.0019879	test: 40.7266263	best: 40.7266263 (0)	total: 679ms	remaining: 33m 55s
200:	learn: 0.0019827	test: 40.7262274	best: 40.7262274 (200)	total: 1m 25s	remaining: 19m 53s
400:	learn: 0.0019802	test: 40.7260536	best: 40.7260536 (400)	total: 2m 51s	remaining: 18m 29s
600:	learn: 0.0019780	test: 40.7258950	best: 40.7258947 (579)	total: 4m 15s	remaining: 16m 58s


In [ ]:
print("catboost scores:", [round(s, 4) for s in cat_result["fold_scores"]])
print(f"mean={np.mean(cat_result['fold_scores']):.4f}  std={np.std(cat_result['fold_scores']):.4f}")

### LightGBM

In [ ]:
lgb_result = train_lightgbm_cv(
    train_df=train,
    test_df=test,
    feature_cols=FEATURE_COLS,
    cat_cols=CAT_COLS,
    target_col=TARGET_COL,
    weight_col=WEIGHT_COL,
    splits=splits
)

In [ ]:
print("lightgbm scores:", [round(s, 4) for s in lgb_result["fold_scores"]])
print(f"mean={np.mean(lgb_result['fold_scores']):.4f}  std={np.std(lgb_result['fold_scores']):.4f}")

In [ ]:
lgb_breakdown = lgb_result["breakdown"]
lgb_breakdown.head(20)

In [ ]:
summary = pd.DataFrame({
    "model": ["CatBoost", "LightGBM"],
    "cv_mean": [np.mean(cat_result["fold_scores"]), np.mean(lgb_result["fold_scores"])],
    "cv_std":  [np.std(cat_result["fold_scores"]), np.std(lgb_result["fold_scores"])],
    "oof_score": [cat_result["overall_score"], lgb_result["overall_score"]]
})

summary = summary.sort_values("oof_score", ascending=False)
summary

## Blend & Select Best Model

In [ ]:
blend_oof  = 0.5 * cat_result["oof_pred"]  + 0.5 * lgb_result["oof_pred"]
blend_test = 0.5 * cat_result["test_pred"] + 0.5 * lgb_result["test_pred"]

blend_score = weighted_rmse_score(train[TARGET_COL], blend_oof, train[WEIGHT_COL])
print(f"blend oof: {blend_score:.4f}")

## Per-Horizon Model (Optional)

Trains a separate LightGBM model for each forecasting horizon. Useful if horizons have very different patterns.

In [ ]:
def train_per_horizon_lightgbm(train_df, test_df, feature_cols, cat_cols, target_col, weight_col):
    oof_pred  = np.zeros(len(train_df))
    test_pred = np.zeros(len(test_df))
    horizon_scores = {}

    for h in sorted(train_df["horizon"].astype(str).unique()):
        print(f"\n--- horizon {h} ---")

        train_mask = train_df["horizon"].astype(str) == h
        test_mask  = test_df["horizon"].astype(str)  == h

        train_h = train_df[train_mask].copy()
        test_h  = test_df[test_mask].copy()

        if len(train_h) == 0 or len(test_h) == 0:
            continue

        splits_h = make_forward_splits(train_h[TIME_COL], n_splits=N_SPLITS, min_train_frac=MIN_TRAIN_FRAC)
        if not splits_h:
            print(f"skipping horizon {h}, not enough data to split")
            continue

        result = train_lightgbm_cv(train_h, test_h, feature_cols, cat_cols, target_col, weight_col, splits_h)

        oof_pred[train_mask] = result["oof_pred"]
        test_pred[test_mask] = result["test_pred"]
        horizon_scores[h]    = result["overall_score"]

    overall_score = weighted_rmse_score(train_df[target_col], oof_pred, train_df[weight_col])
    print(f"\noverall oof: {overall_score:.4f}")
    print("per-horizon:", horizon_scores)

    return {
        "oof_pred":       oof_pred,
        "test_pred":      test_pred,
        "horizon_scores": horizon_scores,
        "overall_score":  overall_score
    }

In [ ]:
candidates = {
    "catboost":    cat_result["overall_score"],
    "lightgbm":    lgb_result["overall_score"],
    "blend_50_50": blend_score,
}

best_name = max(candidates, key=candidates.get)
print("scores:", candidates)
print("picking:", best_name)

if best_name == "catboost":
    final_test_pred = cat_result["test_pred"]
elif best_name == "lightgbm":
    final_test_pred = lgb_result["test_pred"]
else:
    final_test_pred = blend_test

## Create Submission

In [ ]:
submission = pd.DataFrame({
    ID_COL: test[ID_COL].values,
    "prediction": final_test_pred
})

submission.head()

In [ ]:
submission.to_csv("submission.csv", index=False)
print("saved submission.csv")